# Microsoft Fabric — External Repository Library
# `mimesis` 19.1.0 — Fake Data Generation for Testing & Development

> **Author:** Ram Reddy Myla
> **Platform:** Microsoft Fabric (Runtime 1.3 — Spark 3.5)  
> **Library:** mimesis 19.1.0 (installed via External Repositories in Fabric Environment)  
> **Docs:** https://mimesis.name

---

## What is `mimesis`?

Mimesis is a high-performance fake data generator for Python, capable of rapidly producing large volumes of synthetic data for various use cases.

| Feature | Detail |
|---------|--------|
| Speed | Widely recognized as the fastest data generator — approximately 12× faster than Faker |
| Locales | Supports 46 different locales (EN, DE, FR, JA, ZH, etc.) |
| Providers | Person, Address, Finance, Datetime, Internet, Numeric, Text, Code and more |
| Schema-based | Generate structured records of any complexity in one call |
| Relational data | Supports references between schemas for foreign-key style data |

## Why Use It in Fabric?

| Use Case | What you generate |
|----------|------------------|
| Populate dev/test Lakehouse | Realistic customer, order, product tables |
| Pipeline validation testing | Known-shape data to test your `pipeline_validator.py` |
| Load testing | Generate 1M+ rows to stress-test Spark jobs |
| Data anonymization | Replace PII with realistic-looking fake values |
| Demo environments | Show Power BI reports without real customer data |

---

## IMPORTANT — v19.0.0 Breaking Change

> Starting from version 19.0.0, Mimesis has dropped support for builtin providers.  
> This means the old `Generic()` one-object pattern no longer works.  
> You must import individual providers: `Person`, `Address`, `Finance`, etc.

```python
# OLD WAY — does NOT work in v19+
from mimesis import Generic
g = Generic()
g.person.full_name()    # ← AttributeError

# CORRECT WAY — v19+
from mimesis import Person
p = Person()
p.full_name()           # ✅
```

---

## Table of Contents

| Section | Topic |
|---------|-------|
| 1 | Verify installation + all providers |
| 2 | Core providers — Person, Address, Finance, Datetime, Numeric |
| 3 | `Field` — single record generation |
| 4 | `Fieldset` — batch generation (most efficient) |
| 5 | Seeding — reproducible data for testing |
| 6 | Locales — multilingual data |
| 7 | Use case 1 — Populate Lakehouse customer table |
| 8 | Use case 2 — Generate sales transactions |
| 9 | Use case 3 — Large volume load testing (1M rows) |
| 10 | Use case 4 — Relational data (customers + orders) |
| 11 | Use case 5 — Data anonymization (replace PII) |
| 12 | Use case 6 — Pipeline validator test data |


---
## SECTION 1 — Verify Installation

In [1]:
# Cell 1 — Verify mimesis is installed from External Repositories
%pip list | grep mimesis
# Expected: mimesis    19.1.0

StatementMeta(, 76b74b8e-ac6c-40a7-bb74-3004d1754cc9, 5, Finished, Available, Finished, False)

mimesis                            19.1.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 2 — Import all providers we will use throughout this notebook

from mimesis import (
    Person,     # names, email, phone, occupation, birthdate
    Address,    # city, country, state, postal code
    Finance,    # company, price, currency
    Datetime,   # date, timestamp, time
    Internet,   # ip, uri, username
    Numeric,    # integers, floats
    Text,       # words, sentences, titles
    Code,       # ISBN, EAN, IMEI barcodes
    Field,      # single structured record generator
    Fieldset,   # batch structured record generator (FASTEST)
)
from mimesis.locales import Locale
from mimesis.enums import Gender, TimestampFormat
import random

print("All mimesis imports successful")
print("Ready to generate fake data!")

StatementMeta(, 76b74b8e-ac6c-40a7-bb74-3004d1754cc9, 6, Finished, Available, Finished, False)

All mimesis imports successful
Ready to generate fake data!


---
## SECTION 2 — Core Providers

Each provider is a class you instantiate once and call repeatedly.
Pass a `Locale` to get localised data. Default is English.

In [3]:
# Cell 3 — Person provider
# Generates personal information

p = Person(Locale.EN)

print("full_name()          :", p.full_name())
print("first_name()         :", p.first_name())
print("last_name()          :", p.last_name())
print("email()              :", p.email())
print("email(domains)       :", p.email(domains=['company.com', 'corp.org']))
print("telephone()          :", p.telephone())
print("telephone(mask)      :", p.telephone(mask='+1-###-###-####'))
print("username()           :", p.username())
print("occupation()         :", p.occupation())
print("birthdate()          :", p.birthdate(min_year=1970, max_year=2005))
print("gender()             :", p.gender())
print("blood_type()         :", p.blood_type())
print("nationality()        :", p.nationality())
print("university()         :", p.university())
print("password()           :", p.password(length=12))
print("full_name(Female)    :", p.full_name(gender=Gender.FEMALE))
print("full_name(Male)      :", p.full_name(gender=Gender.MALE))

StatementMeta(, 76b74b8e-ac6c-40a7-bb74-3004d1754cc9, 7, Finished, Available, Finished, False)

full_name()          : Arie Raymond
first_name()         : Shin
last_name()          : Mueller
email()              : conditional1998@example.com
email(domains)       : monitor2075@company.com
telephone()          : +12065501844
telephone(mask)      : +1-699-456-6432
username()           : episodes_1813
occupation()         : Weighbridge Operator
birthdate()          : 1977-04-17
gender()             : Other
blood_type()         : AB−
nationality()        : Mexican
university()         : Morehead State University
password()           : W!/YhK|aSp!Y
full_name(Female)    : Lynna Garrett
full_name(Male)      : Beau Strickland


In [4]:
# Cell 4 — Address provider

a = Address(Locale.EN)

print("city()               :", a.city())
print("state()              :", a.state())
print("country()            :", a.country())
print("postal_code()        :", a.postal_code())
print("address()            :", a.address())
print("calling_code()       :", a.calling_code())
print("continent()          :", a.continent())
print("coordinates()        :", a.coordinates())

StatementMeta(, 76b74b8e-ac6c-40a7-bb74-3004d1754cc9, 8, Finished, Available, Finished, False)

city()               : Perris
state()              : Montana
country()            : Burundi
postal_code()        : 44356
address()            : 419 Lick Gardens
calling_code()       : +63
continent()          : Antarctica
coordinates()        : {'longitude': 177.361627, 'latitude': 79.297795}


In [5]:
# Cell 5 — Finance provider

f = Finance(Locale.EN)

print("company()            :", f.company())
print("price()              :", f.price())
print("price(min, max)      :", f.price(minimum=100.0, maximum=9999.99))
print("currency_symbol()    :", f.currency_symbol())
print("currency_iso_code()  :", f.currency_iso_code())
print("stock_ticker()       :", f.stock_ticker())

StatementMeta(, 76b74b8e-ac6c-40a7-bb74-3004d1754cc9, 9, Finished, Available, Finished, False)

company()            : Bain Capital
price()              : 1142.89
price(min, max)      : 4385.59
currency_symbol()    : $
currency_iso_code()  : USD
stock_ticker()       : PDLI


In [6]:
# Cell 6 — Datetime and Numeric providers

d = Datetime(Locale.EN)
n = Numeric()

print("=== Datetime ===")
print("date()               :", d.date())
print("date(start, end)     :", d.date(start=2022, end=2024))
print("timestamp(POSIX)     :", d.timestamp(fmt=TimestampFormat.POSIX))
print("time()               :", d.time())
print("day_of_week()        :", d.day_of_week())
print("month()              :", d.month())
print("year()               :", d.year(minimum=2000, maximum=2024))

print("\n=== Numeric ===")
print("integer_number()     :", n.integer_number(start=1, end=1000))
print("float_number()       :", n.float_number(start=0.0, end=500.0, precision=2))
print("decimal_number()     :", n.decimal_number(start=0, end=100))

StatementMeta(, 76b74b8e-ac6c-40a7-bb74-3004d1754cc9, 10, Finished, Available, Finished, False)

=== Datetime ===
date()               : 2006-01-25
date(start, end)     : 2022-01-03
timestamp(POSIX)     : 1779000782
time()               : 06:14:55.395331
day_of_week()        : Friday
month()              : July
year()               : 2014

=== Numeric ===
integer_number()     : 637
float_number()       : 149.62
decimal_number()     : 90.4161831844697729820836684666574001312255859375


In [7]:
# Cell 7 — Internet, Text, Code providers

net = Internet()
t   = Text(Locale.EN)
c   = Code()

print("=== Internet ===")
print("ip_v4()              :", net.ip_v4())
print("uri()                :", net.uri())
print("mac_address()        :", net.mac_address())
print("http_status_code()   :", net.http_status_code())
print("user_agent()         :", net.user_agent())

print("\n=== Text ===")
print("word()               :", t.word())
print("sentence()           :", t.sentence())
print("title()              :", t.title())
print("words(quantity=4)    :", t.words(quantity=4))

print("\n=== Code ===")
print("isbn()               :", c.isbn())
print("ean()                :", c.ean())
print("imei()               :", c.imei())

StatementMeta(, 76b74b8e-ac6c-40a7-bb74-3004d1754cc9, 11, Finished, Available, Finished, False)

=== Internet ===
ip_v4()              : 189.1.105.117
uri()                : https://soccer.org/2022/04/18/card-wait-january-ce-browsing-injection-original-these-just-feel-fill
mac_address()        : 00:16:3e:39:f4:58
http_status_code()   : 103
user_agent()         : Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/42.0.0.9895 Safari/537.36

=== Text ===
word()               : republic
sentence()           : The arguments can be primitive data types or compound data types.
title()              : Atoms are used within a program to denote distinguished values.
words(quantity=4)    : ['creation', 'recorded', 'shoe', 'profits']

=== Code ===
isbn()               : 1-94200-712-5
ean()                : 00797069
imei()               : 350890801861576


---
## SECTION 3 — `Field`: Single Structured Record Generator

`Field` lets you call any provider method using a dot-notation string like `"person.full_name"`.  
Useful when building one record at a time.

In [8]:
# Cell 8 — Field: dot-notation access to any provider method

field = Field(Locale.EN)

# Call any provider.method using string notation
print(field('person.full_name'))
print(field('person.email'))
print(field('person.telephone'))
print(field('address.city'))
print(field('address.country'))
print(field('finance.company'))
print(field('finance.price', minimum=100, maximum=5000))
print(field('datetime.date', start=2020, end=2024))
print(field('numeric.integer_number', start=1, end=100))

# Build a single structured record
record = {
    'id'        : field('numeric.integer_number', start=1000, end=9999),
    'name'      : field('person.full_name'),
    'email'     : field('person.email', domains=['acme.com']),
    'city'      : field('address.city'),
    'company'   : field('finance.company'),
    'salary'    : field('finance.price', minimum=30000, maximum=200000),
    'hired_date': str(field('datetime.date', start=2015, end=2024)),
}

print("\nSingle structured record:")
for k, v in record.items():
    print(f"  {k:<12} : {v}")

StatementMeta(, 76b74b8e-ac6c-40a7-bb74-3004d1754cc9, 12, Finished, Available, Finished, False)

Glenn Barton
enemy2099@outlook.com
+13611259312
Wilkes-Barre
Suriname
Rockstar Games
406.88
2024-02-27
13

Single structured record:
  id           : 9896
  name         : Myles Morrow
  email        : invalid1878@acme.com
  city         : Sheboygan
  company      : Ameriprise Financial
  salary       : 138528.37
  hired_date   : 2018-11-03


---
## SECTION 4 — `Fieldset`: Batch Generation (Most Efficient)

`Fieldset` generates a list of `i` values per call — ideal for building DataFrames directly.  
This is the recommended approach for generating large datasets.

In [9]:
# Cell 9 — Fieldset: batch generation for DataFrames

# i = number of rows to generate
fs = Fieldset(Locale.EN, i=5)

# Each call returns a LIST of 5 values
names   = fs('person.full_name')
emails  = fs('person.email')
cities  = fs('address.city')
prices  = fs('finance.price', minimum=50, maximum=5000)
dates   = fs('datetime.date', start=2022, end=2024)

print("names  :", names)
print("emails :", emails)
print("cities :", cities)
print("prices :", prices)
print("dates  :", dates)

# Combine into a PySpark DataFrame directly
rows = list(zip(names, emails, cities, [float(p) for p in prices], [str(d) for d in dates]))
df   = spark.createDataFrame(rows, ["name", "email", "city", "price", "date"])
df.show(truncate=False)

StatementMeta(, 76b74b8e-ac6c-40a7-bb74-3004d1754cc9, 13, Finished, Available, Finished, False)

names  : ['Theo Hurley', 'Shalon Hickman', 'Jonathon Porter', 'Jamey Pope', 'Camelia Stafford']
emails : ['installing2070@protonmail.com', 'interstate1941@gmail.com', 'broadcast1884@live.com', 'geographical1957@example.com', 'experienced1886@example.com']
cities : ['Macomb', 'Petaluma', 'Gilroy', 'Springdale', 'Mesa']
prices : [1103.49, 1033.19, 1902.46, 701.61, 4017.32]
dates  : [datetime.date(2024, 11, 28), datetime.date(2022, 3, 4), datetime.date(2024, 11, 26), datetime.date(2022, 8, 24), datetime.date(2024, 5, 6)]
+----------------+-----------------------------+----------+-------+----------+
|name            |email                        |city      |price  |date      |
+----------------+-----------------------------+----------+-------+----------+
|Theo Hurley     |installing2070@protonmail.com|Macomb    |1103.49|2024-11-28|
|Shalon Hickman  |interstate1941@gmail.com     |Petaluma  |1033.19|2022-03-04|
|Jonathon Porter |broadcast1884@live.com       |Gilroy    |1902.46|2024-11-26|
|J

---
## SECTION 5 — Seeding: Reproducible Test Data

Use `seed` when you need the **same data every run** — critical for unit tests  
and regression testing where you compare output across runs.

In [10]:
# Cell 10 — Seeding: same seed = same data every time

# Without seed — different every run
p1 = Person(Locale.EN)
print("No seed run 1:", p1.full_name())
p2 = Person(Locale.EN)
print("No seed run 2:", p2.full_name())    # different name

# With seed — identical every run
p3 = Person(Locale.EN, seed=42)
print("\nSeed=42 run 1:", p3.full_name())
p4 = Person(Locale.EN, seed=42)
print("Seed=42 run 2:", p4.full_name())    # exact same name

# Practical use — seeded dataset for unit tests
p_seed   = Person(Locale.EN, seed=99)
a_seed   = Address(Locale.EN, seed=99)
f_seed   = Finance(Locale.EN, seed=99)
d_seed   = Datetime(Locale.EN, seed=99)
n_seed   = Numeric(seed=99)

test_rows = []
for i in range(4):
    test_rows.append((
        1000 + i,
        p_seed.full_name(),
        p_seed.email(),
        a_seed.city(),
        float(f_seed.price(minimum=100, maximum=5000)),
        str(d_seed.date(start=2023, end=2024)),
    ))

test_df = spark.createDataFrame(
    test_rows,
    ["id", "name", "email", "city", "amount", "date"]
)
print("\nSeeded test dataset (identical every run):")
test_df.show(truncate=False)

StatementMeta(, 76b74b8e-ac6c-40a7-bb74-3004d1754cc9, 14, Finished, Available, Finished, False)

No seed run 1: Quyen Boone
No seed run 2: Tijuana Pace

Seed=42 run 1: Anthony Reilly
Seed=42 run 2: Anthony Reilly

Seeded test dataset (identical every run):
+----+--------------+-------------------------+--------------------+-------+----------+
|id  |name          |email                    |city                |amount |date      |
+----+--------------+-------------------------+--------------------+-------+----------+
|1000|Myung Cote    |grocery1868@yandex.com   |League City         |2079.49|2024-07-07|
|1001|Heath Ramirez |train1845@protonmail.com |La Cañada Flintridge|1080.37|2023-04-08|
|1002|Honey Holden  |paying1908@yahoo.com     |Douglas             |976.13 |2023-02-09|
|1003|Rayford Norton|widespread1842@yandex.com|Phenix City         |1317.31|2024-09-22|
+----+--------------+-------------------------+--------------------+-------+----------+



---
## SECTION 6 — Locales: Multilingual Data

Generate data in 46 locales — useful for testing international pipelines.

In [11]:
# Cell 11 — Generate data in multiple locales

locale_map = {
    'English (US)' : Locale.EN,
    'German'       : Locale.DE,
    'French'       : Locale.FR,
    'Japanese'     : Locale.JA,
    'Russian'      : Locale.RU,
    'Portuguese'   : Locale.PT,
    'Spanish'      : Locale.ES,
}

print(f"{'Locale':<18} {'Name':<25} {'City':<20} {'Company'}")
print("-" * 80)
for locale_name, locale_val in locale_map.items():
    p_loc = Person(locale_val)
    a_loc = Address(locale_val)
    f_loc = Finance(locale_val)
    print(f"{locale_name:<18} {p_loc.full_name():<25} "
          f"{a_loc.city():<20} {f_loc.company()}")

StatementMeta(, 76b74b8e-ac6c-40a7-bb74-3004d1754cc9, 15, Finished, Available, Finished, False)

Locale             Name                      City                 Company
--------------------------------------------------------------------------------
English (US)       Timothy Chang             Pearl                YASH Technologies
German             Barbara Beyersdorf        Arnstein             Freudenberg Group
French             Ibrahim Desnoyers         Reims                Compagnie générale transaérienne
Japanese           はるか 大川                    大和市                  エイブル
Russian            Потап Белова              Велиж                ГК «Титан»
Portuguese         Romã Guerreiro            Anadia               Unicer
Spanish            Meagens Navarro           Tarragona spain      Fiesta hotels


---
## SECTION 7 — Use Case 1: Populate Lakehouse Customer Table

Generate a realistic customer master table and save it to your default Lakehouse.

In [12]:
# Cell 12 — Generate 500 customer records and save to Lakehouse

NUM_CUSTOMERS = 500

p = Person(Locale.EN, seed=1001)
a = Address(Locale.EN, seed=1001)
f = Finance(Locale.EN, seed=1001)
d = Datetime(Locale.EN, seed=1001)
n = Numeric(seed=1001)

segments   = ['Premium', 'Standard', 'Basic', 'Trial']
channels   = ['Online', 'In-Store', 'Mobile App', 'Partner']

customer_rows = []
for i in range(NUM_CUSTOMERS):
    customer_rows.append((
        f'CUST-{10000 + i}',                              # customer_id
        p.full_name(),                                    # name
        p.email(domains=['gmail.com','outlook.com','yahoo.com']),  # email
        p.telephone(mask='+1-###-###-####'),              # phone
        a.city(),                                         # city
        a.state(),                                        # state
        a.country(),                                      # country
        a.postal_code(),                                  # postal_code
        random.choice(segments),                          # segment
        random.choice(channels),                          # acquisition_channel
        str(d.date(start=2018, end=2024)),                # signup_date
        round(float(f.price(minimum=0, maximum=50000)), 2), # lifetime_value
    ))

customer_df = spark.createDataFrame(
    customer_rows,
    ["customer_id", "name", "email", "phone", "city", "state",
     "country", "postal_code", "segment", "acquisition_channel",
     "signup_date", "lifetime_value"]
)

# Save to Lakehouse
customer_df.write.format("delta").mode("overwrite").saveAsTable("dim_customers")

print(f"Generated {customer_df.count()} customer records")
print(f"Saved to Lakehouse → dim_customers")
customer_df.show(5, truncate=True)

# Quick profile
print("\nSegment distribution:")
customer_df.groupBy("segment").count().orderBy("segment").show()
print("Acquisition channels:")
customer_df.groupBy("acquisition_channel").count().orderBy("acquisition_channel").show()

StatementMeta(, 76b74b8e-ac6c-40a7-bb74-3004d1754cc9, 16, Finished, Available, Finished, False)

Generated 500 customer records
Saved to Lakehouse → dim_customers
+-----------+---------------+--------------------+---------------+-----------------+--------------+------------+-----------+--------+-------------------+-----------+--------------+
|customer_id|           name|               email|          phone|             city|         state|     country|postal_code| segment|acquisition_channel|signup_date|lifetime_value|
+-----------+---------------+--------------------+---------------+-----------------+--------------+------------+-----------+--------+-------------------+-----------+--------------+
| CUST-10000| Eugenio Suarez|visual1882@gmail.com|+1-436-456-0779|         Victoria|      Arkansas|    Slovakia|      78097| Premium|           In-Store| 2024-01-25|      39832.55|
| CUST-10001|  Sam Rasmussen| diane2088@yahoo.com|+1-864-182-5053|     Lee's Summit|    Washington|      Guinea|      57718|   Basic|         Mobile App| 2024-04-28|       2931.31|
| CUST-10002| Markita Cherry|

---
## SECTION 8 — Use Case 2: Generate Sales Transactions

Generate a `fact_orders` table that references the customer IDs from `dim_customers`.  
This is **relational data generation** — foreign key relationships preserved.

In [13]:
# Cell 13 — Generate 2000 sales transactions with FK to dim_customers

NUM_ORDERS = 2000

f2 = Finance(Locale.EN, seed=2002)
d2 = Datetime(Locale.EN, seed=2002)
n2 = Numeric(seed=2002)
c2 = Code()

categories   = ['Electronics', 'Clothing', 'Furniture', 'Books',
                 'Sports', 'Kitchen', 'Beauty', 'Toys']
statuses     = ['Completed', 'Pending', 'Shipped', 'Cancelled', 'Returned']
payment_meth = ['Credit Card', 'Debit Card', 'PayPal', 'Bank Transfer']

# Use the customer IDs we already created in dim_customers
customer_ids = [f'CUST-{10000 + i}' for i in range(NUM_CUSTOMERS)]

order_rows = []
for i in range(NUM_ORDERS):
    qty    = n2.integer_number(start=1, end=10)
    price  = round(float(f2.price(minimum=5.0, maximum=2000.0)), 2)
    order_rows.append((
        f'ORD-{100000 + i}',                              # order_id
        random.choice(customer_ids),                      # customer_id (FK)
        f2.company(),                                     # vendor
        random.choice(categories),                        # category
        price,                                            # unit_price
        qty,                                              # quantity
        round(price * qty, 2),                            # total_amount
        random.choice(statuses),                          # status
        random.choice(payment_meth),                      # payment_method
        str(d2.date(start=2023, end=2024)),               # order_date
    ))

orders_df = spark.createDataFrame(
    order_rows,
    ["order_id", "customer_id", "vendor", "category", "unit_price",
     "quantity", "total_amount", "status", "payment_method", "order_date"]
)

orders_df.write.format("delta").mode("overwrite").saveAsTable("fact_orders")

print(f"Generated {orders_df.count()} orders")
print(f"Saved to Lakehouse → fact_orders")
orders_df.show(5, truncate=True)

# Revenue by category
print("\nRevenue by category:")
orders_df.groupBy("category") \
         .agg({"total_amount": "sum", "order_id": "count"}) \
         .withColumnRenamed("sum(total_amount)", "total_revenue") \
         .withColumnRenamed("count(order_id)", "num_orders") \
         .orderBy("total_revenue", ascending=False) \
         .show()

StatementMeta(, 76b74b8e-ac6c-40a7-bb74-3004d1754cc9, 17, Finished, Available, Finished, False)

Generated 2000 orders
Saved to Lakehouse → fact_orders
+----------+-----------+--------------------+-----------+----------+--------+------------+---------+--------------+----------+
|  order_id|customer_id|              vendor|   category|unit_price|quantity|total_amount|   status|payment_method|order_date|
+----------+-----------+--------------------+-----------+----------+--------+------------+---------+--------------+----------+
|ORD-100000| CUST-10036|             Titania|      Books|   1456.34|       1|     1456.34|  Shipped|    Debit Card|2023-09-11|
|ORD-100001| CUST-10462|Brinker Internati...|Electronics|    650.55|       9|     5854.95| Returned|        PayPal|2023-05-29|
|ORD-100002| CUST-10488|              Kroger|   Clothing|   1761.61|       6|    10569.66|  Pending|        PayPal|2024-10-25|
|ORD-100003| CUST-10274|      A+ Investments|   Clothing|   1198.74|      10|     11987.4|Completed| Bank Transfer|2023-02-09|
|ORD-100004| CUST-10407|Union Pacific Rai...|     Sports

---
## SECTION 9 — Use Case 3: Large Volume Load Testing

Generate 1 million rows using `Fieldset` — the most efficient approach.  
Tests Spark's ability to handle large data volumes through your pipeline.

In [14]:
# Cell 14 — Generate 1 million rows for load testing
# Uses Python generators to avoid memory pressure

import time
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

LOAD_TEST_ROWS = 1_000_000
BATCH_SIZE     = 100_000    # generate in batches of 100k to manage memory

schema = StructType([
    StructField("event_id",    StringType(),  False),
    StructField("user_id",     StringType(),  True),
    StructField("event_type",  StringType(),  True),
    StructField("amount",      DoubleType(),  True),
    StructField("ip_address",  StringType(),  True),
    StructField("event_date",  StringType(),  True),
])

event_types = ['click', 'purchase', 'view', 'add_to_cart',
                'remove_from_cart', 'checkout', 'login', 'logout']

start_time = time.time()
all_dfs    = []

for batch_num in range(LOAD_TEST_ROWS // BATCH_SIZE):
    fs  = Fieldset(Locale.EN, i=BATCH_SIZE)
    net = Internet()
    n   = Numeric()
    d   = Datetime()

    rows = list(zip(
        [f'EVT-{batch_num * BATCH_SIZE + j}' for j in range(BATCH_SIZE)],
        [f'USR-{random.randint(1,50000)}'    for _ in range(BATCH_SIZE)],
        [random.choice(event_types)          for _ in range(BATCH_SIZE)],
        [round(float(x), 2) for x in fs('finance.price', minimum=0, maximum=1000)],
        [net.ip_v4()                         for _ in range(BATCH_SIZE)],
        [str(d.date(start=2024, end=2024))   for _ in range(BATCH_SIZE)],
    ))

    batch_df = spark.createDataFrame(rows, schema)
    all_dfs.append(batch_df)
    print(f"  Batch {batch_num + 1}/{LOAD_TEST_ROWS // BATCH_SIZE} generated")

# Union all batches
from functools import reduce
from pyspark.sql import DataFrame
load_test_df = reduce(DataFrame.union, all_dfs)

elapsed = round(time.time() - start_time, 1)
print(f"\nGenerated {load_test_df.count():,} rows in {elapsed}s")

# Save to Lakehouse
load_test_df.write.format("delta").mode("overwrite").saveAsTable("load_test_events")
print(f"Saved to Lakehouse → load_test_events")
load_test_df.show(5)

StatementMeta(, 76b74b8e-ac6c-40a7-bb74-3004d1754cc9, 18, Finished, Available, Finished, False)

  Batch 1/10 generated
  Batch 2/10 generated
  Batch 3/10 generated
  Batch 4/10 generated
  Batch 5/10 generated
  Batch 6/10 generated
  Batch 7/10 generated
  Batch 8/10 generated
  Batch 9/10 generated
  Batch 10/10 generated

Generated 1,000,000 rows in 11.3s
Saved to Lakehouse → load_test_events
+--------+---------+----------------+------+--------------+----------+
|event_id|  user_id|      event_type|amount|    ip_address|event_date|
+--------+---------+----------------+------+--------------+----------+
|   EVT-0|USR-27204|           click|109.56|  52.5.106.244|2024-11-26|
|   EVT-1|USR-12656|            view|254.01|188.193.153.26|2024-08-17|
|   EVT-2|USR-10618|          logout|390.69| 53.39.129.212|2024-06-12|
|   EVT-3|USR-36311|remove_from_cart|487.84|  207.76.4.206|2024-07-30|
|   EVT-4|USR-42621|     add_to_cart|  10.2|36.111.239.139|2024-10-09|
+--------+---------+----------------+------+--------------+----------+
only showing top 5 rows



---
## SECTION 10 — Use Case 4: Data Anonymization

Replace real PII columns in a production DataFrame with realistic fake values.  
The data shape (row count, types, distributions) is preserved — only the values change.

In [15]:
# Cell 15 — Anonymize PII in a DataFrame
# Simulates a production table with real-looking sensitive data

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Simulate a "production" DataFrame with PII
prod_data = [
    (1, "John Smith",    "john.smith@realcompany.com",    "+1-555-123-4567", "123 Main St"),
    (2, "Jane Doe",      "jane.doe@realcompany.com",      "+1-555-234-5678", "456 Oak Ave"),
    (3, "Bob Johnson",   "bob.j@realcompany.com",         "+1-555-345-6789", "789 Pine Rd"),
    (4, "Alice Williams","alice.w@realcompany.com",       "+1-555-456-7890", "321 Elm St"),
    (5, "Charlie Brown", "charlie.b@realcompany.com",     "+1-555-567-8901", "654 Maple Dr"),
]
prod_df = spark.createDataFrame(
    prod_data, ["id", "name", "email", "phone", "address"]
)

print("=== BEFORE anonymization (contains real PII) ===")
prod_df.show(truncate=False)

# Anonymize using mimesis — generate fake replacements
row_count = prod_df.count()
p_anon    = Person(Locale.EN, seed=777)
a_anon    = Address(Locale.EN, seed=777)

anon_names   = [p_anon.full_name()                        for _ in range(row_count)]
anon_emails  = [p_anon.email(domains=['anonymized.com'])  for _ in range(row_count)]
anon_phones  = [p_anon.telephone(mask='+1-###-###-####')  for _ in range(row_count)]
anon_address = [a_anon.address()                          for _ in range(row_count)]

# Collect IDs, zip with fake values, rebuild DataFrame
ids = [row['id'] for row in prod_df.select('id').collect()]

anon_rows = list(zip(ids, anon_names, anon_emails, anon_phones, anon_address))
anon_df   = spark.createDataFrame(anon_rows, ["id", "name", "email", "phone", "address"])

print("=== AFTER anonymization (PII replaced with fake data) ===")
anon_df.show(truncate=False)

# Save anonymized version
anon_df.write.format("delta").mode("overwrite").saveAsTable("dim_customers_anonymized")
print("Saved to Lakehouse → dim_customers_anonymized")

StatementMeta(, 76b74b8e-ac6c-40a7-bb74-3004d1754cc9, 19, Finished, Available, Finished, False)

=== BEFORE anonymization (contains real PII) ===
+---+--------------+--------------------------+---------------+------------+
|id |name          |email                     |phone          |address     |
+---+--------------+--------------------------+---------------+------------+
|1  |John Smith    |john.smith@realcompany.com|+1-555-123-4567|123 Main St |
|2  |Jane Doe      |jane.doe@realcompany.com  |+1-555-234-5678|456 Oak Ave |
|3  |Bob Johnson   |bob.j@realcompany.com     |+1-555-345-6789|789 Pine Rd |
|4  |Alice Williams|alice.w@realcompany.com   |+1-555-456-7890|321 Elm St  |
|5  |Charlie Brown |charlie.b@realcompany.com |+1-555-567-8901|654 Maple Dr|
+---+--------------+--------------------------+---------------+------------+

=== AFTER anonymization (PII replaced with fake data) ===
+---+---------------+-------------------------------+---------------+--------------------------+
|id |name           |email                          |phone          |address                   |
+---+

---
## SECTION 11 — Use Case 5: Pipeline Validator Test Data

Generate a **known-shape, known-content** dataset with controlled DQ issues  
so you can test your `pipeline_validator.py` checks deterministically.

In [16]:
# Cell 16 — Generate controlled test data for pipeline_validator
# Uses seed=42 so results are identical every test run

from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType, DateType
)

p_t  = Person(Locale.EN, seed=42)
a_t  = Address(Locale.EN, seed=42)
f_t  = Finance(Locale.EN, seed=42)
d_t  = Datetime(Locale.EN, seed=42)
n_t  = Numeric(seed=42)

categories = ['Electronics', 'Clothing', 'Furniture', 'Books', 'Sports']

test_data = []
for i in range(8):
    test_data.append((
        1001 + i,
        p_t.full_name(),
        random.choice(categories),
        round(float(f_t.price(minimum=50.0, maximum=5000.0)), 2),
        n_t.integer_number(start=1, end=20),
        str(d_t.date(start=2024, end=2024)),
    ))

# Inject controlled DQ issues for testing:
test_data.append((1009, p_t.full_name(), 'Electronics', None,    1, '2024-01-10'))  # null amount
test_data.append((1001, test_data[0][1], test_data[0][2],        # duplicate of row 0
                  test_data[0][3], test_data[0][4], test_data[0][5]))

test_schema = StructType([
    StructField("order_id",   IntegerType(), False),
    StructField("customer",   StringType(),  True),
    StructField("category",   StringType(),  True),
    StructField("amount",     DoubleType(),  True),
    StructField("quantity",   IntegerType(), True),
    StructField("order_date", StringType(),  True),
])

test_df = spark.createDataFrame(test_data, test_schema)
test_df.write.format("delta").mode("overwrite").saveAsTable("test_sales_data")

print(f"Test dataset: {test_df.count()} rows")
print(f"Known issues: 1 null (amount), 1 duplicate (order_id 1001)")
print(f"Saved to Lakehouse → test_sales_data")
test_df.show(truncate=False)

StatementMeta(, 76b74b8e-ac6c-40a7-bb74-3004d1754cc9, 20, Finished, Available, Finished, False)

Test dataset: 10 rows
Known issues: 1 null (amount), 1 duplicate (order_id 1001)
Saved to Lakehouse → test_sales_data
+--------+----------------+-----------+-------+--------+----------+
|order_id|customer        |category   |amount |quantity|order_date|
+--------+----------------+-----------+-------+--------+----------+
|1001    |Anthony Reilly  |Sports     |3215.16|4       |2024-01-24|
|1002    |Kai Day         |Clothing   |173.8  |1       |2024-04-08|
|1003    |Cleveland Osborn|Furniture  |1411.4 |9       |2024-12-04|
|1004    |Zack Holder     |Clothing   |1154.89|8       |2024-10-14|
|1005    |Arden Brady     |Books      |3695.53|8       |2024-01-03|
|1006    |Gilberto Lane   |Furniture  |3399.66|5       |2024-04-17|
|1007    |Vern Cortez     |Furniture  |4466.29|4       |2024-09-07|
|1008    |Jaye Hunt       |Sports     |480.35 |18      |2024-04-15|
|1009    |Aleen Robbins   |Electronics|NULL   |1       |2024-01-10|
|1001    |Anthony Reilly  |Sports     |3215.16|4       |2024-01-24

In [20]:
# Cell 17 — Write test data to Warehouse and run pipeline validation
# This is the full end-to-end test using mimesis data

import com.microsoft.spark.fabric
from pipeline_validator import PipelineValidator

WAREHOUSE_NAME  = "b20260311_dwh"    # your Warehouse name
WAREHOUSE_TABLE = f"{WAREHOUSE_NAME}.dbo.test_sales_data"

# Read source from Lakehouse
lh_df = spark.sql("SELECT * FROM test_sales_data")

# Write to Warehouse using synapsesql
lh_df.write.mode("overwrite").synapsesql(WAREHOUSE_TABLE)

# Read back from Warehouse
wh_df = spark.read.synapsesql(WAREHOUSE_TABLE)

# Run pipeline validation on mimesis-generated test data
pv = PipelineValidator(spark, 
    source_df    = lh_df,
    target_df    = wh_df,
    source_label = "Test Lakehouse (mimesis)",
    target_label = "Test Warehouse",
    key_cols     = ["order_id"],
    numeric_cols = ["amount", "quantity"],
)
pv.run_all_checks()
# Expected: Row count PASS, nulls FAIL (1 null amount), duplicates FAIL (1 duplicate)

StatementMeta(, 76b74b8e-ac6c-40a7-bb74-3004d1754cc9, 24, Finished, Available, Finished, False)


############################################################
  PIPELINE VALIDATOR — Microsoft Fabric
  Source : Test Lakehouse (mimesis)
  Target : Test Warehouse
  Run at : 2026-03-19 04:06:00
############################################################

  CHECK 1 — Row Count Reconciliation
  [INFO]  Test Lakehouse (mimesis) rows : 10
  [INFO]  Test Warehouse rows : 10
  [PASS]  Row counts MATCH

  CHECK 2 — Column Count Reconciliation
  [INFO]  Test Lakehouse (mimesis) columns : 6
  [INFO]  Test Warehouse columns : 6
  [PASS]  Column counts MATCH

  CHECK 3 — Schema Match
  [PASS]  Schema MATCHES — all columns and types align

  CHECK 4 — Null Count Check
  [INFO]  Test Lakehouse (mimesis) — total nulls: 1
          [OK]  order_id: 0
          [OK]  customer: 0
          [OK]  category: 0
          [!!]  amount: 1
          [OK]  quantity: 0
          [OK]  order_date: 0

  [INFO]  Test Warehouse — total nulls: 1
          [OK]  quantity: 0
          [OK]  order_id: 0
          [!!]

---
## SECTION 12 — Quick Reference

### All Providers and Key Methods

```python
from mimesis import Person, Address, Finance, Datetime, Internet, Numeric, Text, Code
from mimesis import Field, Fieldset
from mimesis.locales import Locale
from mimesis.enums import Gender, TimestampFormat

# ── Person ────────────────────────────────────────────────────
p = Person(Locale.EN)
p.full_name()                              # 'John Smith'
p.full_name(gender=Gender.FEMALE)          # 'Jane Smith'
p.email(domains=['company.com'])           # 'john1990@company.com'
p.telephone(mask='+1-###-###-####')        # '+1-555-123-4567'
p.birthdate(min_year=1970, max_year=2000)  # datetime.date(1985, 3, 12)
p.occupation()                             # 'Software Engineer'
p.username()                               # 'john_smith_99'
p.password(length=16)                      # 'xK9!mN2@pQ7#rT4$'

# ── Address ───────────────────────────────────────────────────
a = Address(Locale.EN)
a.city()                                   # 'Austin'
a.state()                                  # 'Texas'
a.country()                                # 'United States'
a.postal_code()                            # '78701'
a.address()                                # '123 Main St'
a.coordinates()                            # {'latitude': 30.26, 'longitude': -97.74}

# ── Finance ───────────────────────────────────────────────────
f = Finance(Locale.EN)
f.company()                                # 'Acme Corporation'
f.price(minimum=10.0, maximum=9999.99)     # 4231.87
f.currency_iso_code()                      # 'USD'
f.stock_ticker()                           # 'AAPL'

# ── Datetime ──────────────────────────────────────────────────
d = Datetime()
d.date(start=2020, end=2024)               # datetime.date(2022, 6, 15)
d.timestamp(fmt=TimestampFormat.POSIX)     # 1718992237
d.time()                                   # datetime.time(14, 32, 17)

# ── Numeric ───────────────────────────────────────────────────
n = Numeric()
n.integer_number(start=1, end=1000)        # 427
n.float_number(start=0.0, end=100.0, precision=2)  # 73.41

# ── Field (single record) ─────────────────────────────────────
field = Field(Locale.EN)
field('person.full_name')                  # 'John Smith'
field('finance.price', minimum=100, maximum=5000)  # 2341.87

# ── Fieldset (batch — most efficient for DataFrames) ──────────
fs = Fieldset(Locale.EN, i=1000)           # generates 1000 values per call
names  = fs('person.full_name')            # list of 1000 names
prices = fs('finance.price', minimum=10, maximum=500)  # list of 1000 prices

# ── Seeding (reproducible data for tests) ─────────────────────
p = Person(Locale.EN, seed=42)             # same data every run
```

---

### When to Use What

| Need | Use |
|------|-----|
| One field at a time | Individual provider: `Person().full_name()` |
| One structured record | `Field` with dot notation |
| Many records for a DataFrame | `Fieldset` with `i=row_count` |
| Same data every test run | Any provider with `seed=42` |
| Localised data | `Person(Locale.DE)` etc. |
| 1M+ rows (load test) | `Fieldset` in batches + `DataFrame.union` |
| Anonymize existing data | Generate replacements, zip with original IDs |

---

### Official Reference Links

| Resource | URL |
|----------|-----|
| Mimesis documentation | https://mimesis.name |
| API reference | https://mimesis.name/master/api.html |
| Schema/Fieldset guide | https://mimesis.name/latest/schema.html |
| GitHub | https://github.com/lk-geimfari/mimesis |
| PyPI | https://pypi.org/project/mimesis/ |
| Fabric External Libraries | https://learn.microsoft.com/en-us/fabric/data-engineering/environment-manage-library |

---
*Notebook created by Myla Ram Reddy | Microsoft Fabric — mimesis External Repository Teaching Reference*
